# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** Diyanti Pratiwi
- **Email:** [Isi dengan Email Terdaftar Dicoding Anda]
- **ID Dicoding:** [Isi dengan Username/ID Dicoding Anda]

## Menentukan Pertanyaan Bisnis

Perumusan pertanyaan bisnis pada proyek ini menggunakan kerangka **SMART** (*Specific, Measurable, Action-Oriented, Relevant, Time-bound*) agar hasil analisis memiliki arah bisnis yang terukur dan dapat ditindaklanjuti:

- **Pertanyaan 1:** Bagaimana perbandingan rata-rata dan total penyewaan sepeda antara pengguna *casual* dan *registered* berdasarkan pola jam operasional (*hourly*) pada hari kerja (*working day*) dibandingkan hari libur/akhir pekan (*weekend/holiday*) sepanjang periode 2011–2012, dan pada jam berapa saja puncak lonjakan permintaan terjadi?
  - *Specific:* Memfokuskan analisis pada perbedaan perilaku dua segmen pengguna (*casual* vs *registered*) berdasarkan jam operasional dan status hari (*working day* vs *weekend/holiday*).
  - *Measurable:* Menghitung rata-rata penyewaan per jam, total transaksi, dan proporsi kontribusi masing-masing segmen.
  - *Action-Oriented:* Hasilnya digunakan untuk menentukan jadwal *rebalancing* (distribusi ulang) armada sepeda antar stasiun komuter dan area rekreasi serta waktu terbaik menjalankan promo konversi member.
  - *Relevant:* Ketersediaan sepeda pada jam puncak sangat menentukan tingkat kepuasan pengguna dan optimalisasi pendapatan.
  - *Time-bound:* Dibatasi pada rentang data historis tahun 2011 hingga 2012.

- **Pertanyaan 2:** Seberapa besar pengaruh perubahan kondisi cuaca (`weathersit`) dan musim (`season`) terhadap rata-rata jumlah penyewaan sepeda harian (`cnt`) pada tahun 2011 dan 2012, serta kondisi mana yang menyebabkan penurunan permintaan paling signifikan?
  - *Specific:* Menganalisis dampak langsung empat kategori cuaca dan empat musim terhadap volume penyewaan harian.
  - *Measurable:* Mengukur rata-rata penyewaan harian, persentase penurunan antar kondisi cuaca/musim, serta pertumbuhan tahunan (*Year-over-Year*).
  - *Action-Oriented:* Menjadi acuan penjadwalan perawatan rutin armada (*preventive maintenance*) saat permintaan rendah dan penambahan kapasitas saat cuaca/musim puncak.
  - *Relevant:* Faktor cuaca dan musim adalah variabel eksternal paling krusial dalam operasional transportasi berbasis sepeda.
  - *Time-bound:* Periode pengamatan tahunan 2011–2012.

## Import Semua Packages/Library yang Digunakan

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

## Data Wrangling

### Gathering Data

#### Load `day.csv` dan `hour.csv`

In [2]:
day_df = pd.read_csv('data/day.csv')
hour_df = pd.read_csv('data/hour.csv')

print(f'Dimensi day_df : {day_df.shape[0]} baris, {day_df.shape[1]} kolom')
print(f'Dimensi hour_df: {hour_df.shape[0]} baris, {hour_df.shape[1]} kolom')

print('\n--- 3 Baris Pertama day_df ---')
print(day_df.head(3))
print('\n--- 3 Baris Pertama hour_df ---')
print(hour_df.head(3))

Dimensi day_df : 731 baris, 16 kolom
Dimensi hour_df: 17379 baris, 17 kolom

--- 3 Baris Pertama day_df ---
   instant      dteday  season  yr  mnth  holiday  weekday  workingday  weathersit      temp     atemp       hum  windspeed  casual  registered   cnt
0        1  2011-01-01       1   0     1        0        6           0           2  0.344167  0.363625  0.805833   0.160446     331         654   985
1        2  2011-01-02       1   0     1        0        0           0           2  0.363478  0.353739  0.696087   0.248539     131         670   801
2        3  2011-01-03       1   0     1        0        1           1           1  0.196364  0.189405  0.437273   0.248309     120        1229  1349

--- 3 Baris Pertama hour_df ---
   instant      dteday  season  yr  mnth  hr  holiday  weekday  workingday  weathersit  temp   atemp   hum  windspeed  casual  registered  cnt
0        1  2011-01-01       1   0     1   0        0        6           0           1  0.24  0.2879  0.81        0.

**Insight:**
- Dataset `day.csv` berisi **731 baris** catatan penyewaan sepeda agregasi harian selama 2 tahun penuh (1 Januari 2011 – 31 Desember 2012, mencakup tahun kabisat 2012 yang memiliki 366 hari).
- Dataset `hour.csv` berisi **17.379 baris** catatan agregasi per jam dengan struktur kolom yang identik, ditambah kolom `hr` (jam 0 sampai 23).

### Assessing Data

#### Memeriksa Tipe Data, Missing Value, Duplicate Data, Invalid Value, dan Outlier

In [3]:
print('=== INFO DAY_DF ===')
day_df.info()
print('\n=== INFO HOUR_DF ===')
hour_df.info()

print('\nMissing value day_df :', day_df.isna().sum().sum())
print('Missing value hour_df:', hour_df.isna().sum().sum())
print('Duplicate day_df     :', day_df.duplicated().sum())
print('Duplicate hour_df    :', hour_df.duplicated().sum())

=== INFO DAY_DF ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     731 non-null    int64  
 1   dteday      731 non-null    object 
 2   season      731 non-null    int64  
 3   yr          731 non-null    int64  
 4   mnth        731 non-null    int64  
 5   holiday     731 non-null    int64  
 6   weekday     731 non-null    int64  
 7   workingday  731 non-null    int64  
 8   weathersit  731 non-null    int64  
 9   temp        731 non-null    float64
 10  atemp       731 non-null    float64
 11  hum         731 non-null    float64
 12  windspeed   731 non-null    float64
 13  casual      731 non-null    int64  
 14  registered  731 non-null    int64  
 15  cnt         731 non-null    int64  
dtypes: float64(4), int64(11), object(1)
memory usage: 91.5+ KB

=== INFO HOUR_DF ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1

In [4]:
print('=== STATISTIK DESKRIPTIF DAY_DF ===')
print(day_df.describe().round(3))
print('\n=== STATISTIK DESKRIPTIF HOUR_DF ===')
print(hour_df.describe().round(3))

# Cek konsistensi penjumlahan casual + registered == cnt
print('\nSelisih casual + registered != cnt (day) :', (day_df['casual'] + day_df['registered'] != day_df['cnt']).sum())
print('Selisih casual + registered != cnt (hour):', (hour_df['casual'] + hour_df['registered'] != hour_df['cnt']).sum())

# Cek nilai tidak valid secara fisik (kelembapan == 0)
print('\nData dengan hum == 0 di day_df:')
print(day_df[day_df['hum'] == 0][['instant', 'dteday', 'weathersit', 'temp', 'hum', 'windspeed', 'cnt']])
print('Jumlah baris dengan hum == 0 di hour_df:', (hour_df['hum'] == 0).sum())

=== STATISTIK DESKRIPTIF DAY_DF ===
       instant   season       yr     mnth  holiday  weekday  workingday  weathersit     temp    atemp      hum  windspeed    casual  registered       cnt
count  731.000  731.000  731.000  731.000  731.000  731.000     731.000     731.000  731.000  731.000  731.000    731.000   731.000     731.000   731.000
mean   366.000    2.497    0.501    6.520    0.029    2.997       0.684       1.395    0.495    0.474    0.628      0.190   848.176    3656.172  4504.349
std    211.166    1.111    0.500    3.452    0.167    2.005       0.465       0.545    0.183    0.163    0.142      0.077   686.622    1560.256  1937.211
min      1.000    1.000    0.000    1.000    0.000    0.000       0.000       1.000    0.059    0.079    0.000      0.022     2.000      20.000    22.000
25%    183.500    2.000    0.000    4.000    0.000    1.000       0.000       1.000    0.337    0.338    0.520      0.135   315.500    2497.000  3152.000
50%    366.000    3.000    1.000    7.00

In [5]:
def check_outliers_iqr(df, cols):
    summary = []
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = ((df[col] < lower) | (df[col] > upper)).sum()
        summary.append({'Kolom': col, 'Batas Bawah': round(lower, 3), 'Batas Atas': round(upper, 3), 'Jumlah Outlier': count})
    return pd.DataFrame(summary)

print('Outlier IQR pada day_df:')
print(check_outliers_iqr(day_df, ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']))
print('\nOutlier IQR pada hour_df:')
print(check_outliers_iqr(hour_df, ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']))

Outlier IQR pada day_df:
        Kolom  Batas Bawah  Batas Atas  Jumlah Outlier
0        temp       -0.140       1.133               0
1       atemp       -0.068       1.015               0
2         hum        0.205       1.046               2
3   windspeed       -0.012       0.381              13
4      casual     -855.250    2266.750              44
5  registered     -922.250    8195.750               0
6         cnt    -1054.000   10162.000               0

Outlier IQR pada hour_df:
        Kolom  Batas Bawah  Batas Atas  Jumlah Outlier
0        temp       -0.140       1.140               0
1       atemp       -0.099       1.053               0
2         hum        0.030       1.230              22
3   windspeed       -0.119       0.478             342
4      casual      -62.000     114.000            1192
5  registered     -245.000     499.000             680
6         cnt     -321.500     642.500             505


**Steps to Take:**
1. **Perbaikan Tipe Data (`dteday`):** Mengubah kolom `dteday` pada `day_df` dan `hour_df` dari tipe `object` (string) menjadi `datetime64[ns]`.
2. **Penanganan *Invalid Value* (`hum == 0`):** Pada tanggal `2011-03-10`, tercatat nilai kelembapan `0.0` pada seluruh 22 jam pengamatan padahal kondisi cuaca (`weathersit`) adalah `3` (hujan ringan/salju). Secara meteorologis mustahil terjadi hujan pada kelembapan 0% (indikasi gangguan sensor), sehingga nilai `0.0` akan diimputasi menggunakan median kelembapan pada kondisi cuaca `weathersit == 3`.
3. **Transformasi Label Kategorikal & Skala Aktual:** Mengonversi kolom kode numerik (`season`, `yr`, `mnth`, `weekday`, `workingday`, `weathersit`) menjadi label teks deskriptif serta menambahkan kolom suhu (°C), kelembapan (%), dan kecepatan angin (km/jam) dalam satuan aktual agar analisis dan visualisasi bebas ambiguitas.
4. **Penanganan *Outlier*:** Nilai ekstrem pada `casual`, `registered`, `cnt`, dan `windspeed` merupakan lonjakan alami saat akhir pekan, jam komuter, atau cuaca berangin (bukan kesalahan input), sehingga dipertahankan agar tidak menghilangkan fenomena bisnis sebenarnya.

**Insight:**
- Kedua dataset bersih dari *missing value* eksplisit (`NaN`) dan duplikasi baris, serta penjumlahan `casual + registered` konsisten 100% dengan kolom `cnt`.
- Pemeriksaan validitas domain berhasil menemukan anomali sensor kelembapan `0.0` pada `2011-03-10` serta ketidaktepatan tipe data tanggal.

### Cleaning Data

#### Memperbaiki Tipe Data, Imputasi Nilai Kelembapan Tidak Valid, dan Mapping Label

In [6]:
day_clean = day_df.copy()
hour_clean = hour_df.copy()

# 1. Konversi dteday menjadi datetime
day_clean['dteday'] = pd.to_datetime(day_clean['dteday'])
hour_clean['dteday'] = pd.to_datetime(hour_clean['dteday'])

# 2. Imputasi nilai invalid hum == 0 dengan median hum pada weathersit == 3
median_hum_day = day_clean.loc[day_clean['weathersit'] == 3, 'hum'].median()
median_hum_hour = hour_clean.loc[hour_clean['weathersit'] == 3, 'hum'].median()

day_clean.loc[day_clean['hum'] == 0, 'hum'] = median_hum_day
hour_clean.loc[hour_clean['hum'] == 0, 'hum'] = median_hum_hour

# 3. Mapping label kategorikal & denormalisasi satuan cuaca
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
yr_map = {0: 2011, 1: 2012}
weathersit_map = {
    1: 'Clear / Partly Cloudy',
    2: 'Mist / Cloudy',
    3: 'Light Snow / Light Rain',
    4: 'Heavy Rain / Snow'
}
workingday_map = {0: 'Weekend / Holiday', 1: 'Working Day'}
weekday_map = {0: 'Sunday', 1: 'Monday', 2: 'Tuesday', 3: 'Wednesday', 4: 'Thursday', 5: 'Friday', 6: 'Saturday'}

for df in [day_clean, hour_clean]:
    df['season_label'] = df['season'].map(season_map)
    df['year'] = df['yr'].map(yr_map)
    df['weather_label'] = df['weathersit'].map(weathersit_map)
    df['day_type'] = df['workingday'].map(workingday_map)
    df['weekday_name'] = df['weekday'].map(weekday_map)
    df['temp_c'] = (df['temp'] * 41).round(2)
    df['atemp_c'] = (df['atemp'] * 50).round(2)
    df['hum_pct'] = (df['hum'] * 100).round(2)
    df['windspeed_kmh'] = (df['windspeed'] * 67).round(2)

print('Sisa hum == 0 (day) :', (day_clean['hum'] == 0).sum())
print('Sisa hum == 0 (hour):', (hour_clean['hum'] == 0).sum())
print('Tipe data dteday    :', day_clean['dteday'].dtype)
print(day_clean[['dteday', 'year', 'season_label', 'day_type', 'weather_label', 'temp_c', 'hum_pct', 'casual', 'registered', 'cnt']].head())

Sisa hum == 0 (day) : 0
Sisa hum == 0 (hour): 0
Tipe data dteday    : datetime64[ns]
      dteday  year season_label           day_type          weather_label  temp_c  hum_pct  casual  registered   cnt
0 2011-01-01  2011       Spring  Weekend / Holiday          Mist / Cloudy   14.11    80.58     331         654   985
1 2011-01-02  2011       Spring  Weekend / Holiday          Mist / Cloudy   14.90    69.61     131         670   801
2 2011-01-03  2011       Spring        Working Day  Clear / Partly Cloudy    8.05    43.73     120        1229  1349
3 2011-01-04  2011       Spring        Working Day  Clear / Partly Cloudy    8.20    59.04     108        1454  1562
4 2011-01-05  2011       Spring        Working Day  Clear / Partly Cloudy    9.31    43.70      82        1518  1600


**Insight:**
- Kolom `dteday` kini bertipe `datetime64[ns]`, memungkinkan agregasi deret waktu (*time-series*) yang akurat.
- Nilai kelembapan anomali `0.0` pada `2011-03-10` telah diperbaiki menggunakan median cuaca hujan ringan (`88,83%` pada skala harian dan `87,00%` pada skala per jam), sehingga korelasi variabel cuaca menjadi lebih valid.

## Exploratory Data Analysis (EDA)

### Explore Performa Penyewaan Keseluruhan & Pertumbuhan Tahunan (2011 vs 2012)

In [7]:
total_cnt = day_clean['cnt'].sum()
total_cas = day_clean['casual'].sum()
total_reg = day_clean['registered'].sum()

print(f'Total Penyewaan (2011-2012): {total_cnt:,} transaksi')
print(f'Total Pengguna Registered  : {total_reg:,} ({total_reg/total_cnt*100:.2f}%)')
print(f'Total Pengguna Casual      : {total_cas:,} ({total_cas/total_cnt*100:.2f}%)')

yoy_df = day_clean.groupby('year')[['casual', 'registered', 'cnt']].agg(['sum', 'mean', 'median']).round(2)
print('\nPerbandingan Tahunan (YoY):')
print(yoy_df)

growth_pct = (day_clean[day_clean['year']==2012]['cnt'].sum() - day_clean[day_clean['year']==2011]['cnt'].sum()) / day_clean[day_clean['year']==2011]['cnt'].sum() * 100
print(f'\nPertumbuhan Total Penyewaan YoY (2011 -> 2012): +{growth_pct:.2f}%')

Total Penyewaan (2011-2012): 3,292,679 transaksi
Total Pengguna Registered  : 2,672,662 (81.17%)
Total Pengguna Casual      : 620,017 (18.83%)

Perbandingan Tahunan (YoY):
      casual                 registered                       cnt                 
         sum     mean median        sum     mean  median      sum     mean  median
year                                                                              
2011  247252   677.40  614.0     995851  2728.36  2915.0  1243103  3405.76  3740.0
2012  372765  1018.48  904.5    1676811  4581.45  4776.5  2049576  5599.93  5927.0

Pertumbuhan Total Penyewaan YoY (2011 -> 2012): +64.88%


### Explore Pola Penyewaan Berdasarkan Tipe Hari & Jam (Pertanyaan 1)

In [8]:
daytype_agg = day_clean.groupby('day_type')[['casual', 'registered', 'cnt']].agg(['mean', 'median', 'sum']).round(2)
print('Rata-rata & Total Harian Berdasarkan Tipe Hari:')
print(daytype_agg)

hourly_agg = hour_clean.groupby(['day_type', 'hr'])[['casual', 'registered', 'cnt']].mean().round(2).reset_index()

print('\nTop 5 Jam Tersibuk pada Working Day:')
print(hourly_agg[hourly_agg['day_type'] == 'Working Day'].sort_values(by='cnt', ascending=False).head(5))

print('\nTop 5 Jam Tersibuk pada Weekend / Holiday:')
print(hourly_agg[hourly_agg['day_type'] == 'Weekend / Holiday'].sort_values(by='cnt', ascending=False).head(5))

Rata-rata & Total Harian Berdasarkan Tipe Hari:
                    casual                 registered                       cnt                 
                      mean  median     sum       mean  median      sum     mean  median      sum
day_type                                                                                        
Weekend / Holiday  1371.13  1338.0  316732    2959.03  2928.0   683537  4330.17  4459.0  1000269
Working Day         606.57   616.5  303285    3978.25  3866.0  1989125  4584.82  4582.0  2292410

Top 5 Jam Tersibuk pada Working Day:
       day_type  hr  casual  registered     cnt
41  Working Day  17   56.90      468.39  525.29
42  Working Day  18   50.45      441.78  492.23
32  Working Day   8   22.26      454.74  477.01
43  Working Day  19   41.02      307.38  348.40
40  Working Day  16   48.34      244.78  293.12

Top 5 Jam Tersibuk pada Weekend / Holiday:
             day_type  hr  casual  registered     cnt
13  Weekend / Holiday  13  136.42      236.

### Explore Dampak Kondisi Cuaca & Musim Terhadap Penyewaan (Pertanyaan 2)

In [9]:
weather_day_agg = day_clean.groupby('weather_label')[['casual', 'registered', 'cnt']].agg(['count', 'mean', 'median']).round(2)
print('Ringkasan Penyewaan Harian Berdasarkan Cuaca:')
print(weather_day_agg)

weather_hour_agg = hour_clean.groupby('weather_label')[['casual', 'registered', 'cnt']].agg(['count', 'mean', 'median']).round(2)
print('\nRingkasan Penyewaan per Jam Berdasarkan Cuaca:')
print(weather_hour_agg)

season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_agg = day_clean.groupby('season_label')[['casual', 'registered', 'cnt']].agg(['count', 'mean', 'median', 'sum']).reindex(season_order).round(2)
print('\nRingkasan Penyewaan Harian Berdasarkan Musim:')
print(season_agg)

Ringkasan Penyewaan Harian Berdasarkan Cuaca:
                        casual                registered                    cnt                 
                         count    mean median      count     mean  median count     mean  median
weather_label                                                                                   
Clear / Partly Cloudy      463  964.03  829.0        463  3912.76  3875.0   463  4876.79  4844.0
Light Snow / Light Rain     21  185.48  126.0         21  1617.81  1672.0    21  1803.29  1817.0
Mist / Cloudy              247  687.35  534.0        247  3348.51  3352.0   247  4035.86  4040.0

Ringkasan Penyewaan per Jam Berdasarkan Cuaca:
                        casual               registered                   cnt               
                         count   mean median      count    mean median  count    mean median
weather_label                                                                               
Clear / Partly Cloudy    11413  40.55   20.0 

**Insight:**
- **Pertumbuhan Bisnis Pesat:** Total penyewaan selama 2011–2012 mencapai **3.292.679 kali**, didominasi oleh pelanggan *registered* sebesar **81,17%** (2.672.662) dibandingkan *casual* sebesar **18,83%** (620.017). Volume transaksi tahunan melonjak **+64,88%** dari 1.243.103 (2011) menjadi 2.049.576 (2012).
- **Kontras Perilaku Segmen Pengguna:** Pada *Working Day*, rata-rata penyewaan *registered* mencapai **3.978,25/hari** dengan dua puncak tajam (*bimodal*) pada jam berangkat kerja (**08:00** = 477,01/jam) dan pulang kerja (**17:00** = 525,29/jam, **18:00** = 492,23/jam). Sebaliknya, pada *Weekend/Holiday*, penyewaan *casual* melonjak **+126,05%** (dari 606,57 menjadi **1.371,13/hari**) dengan puncak aktivitas tunggal (*unimodal*) di siang hingga sore hari (**12:00–16:00**, tertinggi pukul **13:00** sebesar 372,73/jam).
- **Sensitivitas Terhadap Cuaca & Musim:** Cuaca cerah (*Clear / Partly Cloudy*) menghasilkan rata-rata **4.876,79 penyewaan/hari**, namun anjlok **-17,24%** saat berkabut/berawan (*Mist / Cloudy*: 4.035,86/hari) dan merosot tajam **-63,02%** saat hujan ringan/salju (*Light Snow / Light Rain*: 1.803,29/hari). Secara musiman, **Fall (Musim Gugur)** mencatat rata-rata tertinggi (**5.644,30/hari**), sedangkan **Spring (Musim Semi)** terendah (**2.604,13/hari**, lebih rendah **53,86%** dibanding Fall).

## Visualization & Explanatory Analysis

Seluruh visualisasi berikut dirancang dengan menerapkan **prinsip desain dan integritas visualisasi data**:
- **High Data-Ink Ratio:** Menghilangkan garis tepi atas/kanan (*spines*) serta menggunakan *grid* tipis agar perhatian langsung tertuju pada pola data.
- **Zero-Baseline & Proporsional (Anti-Misleading):** Sumbu Y pada diagram batang dimulai dari **0** sehingga perbandingan panjang batang mencerminkan kuantitas sebenarnya (*Lie Factor* = 1).
- **Strategic Color Highlighting (Anti-Distracts):** Menggunakan warna netral abu-abu untuk kategori pembanding dan warna aksen kontras hanya untuk menyorot titik puncak atau penurunan paling signifikan.

### Pertanyaan 1: Bagaimana perbandingan rata-rata dan total penyewaan sepeda antara pengguna casual dan registered berdasarkan pola jam operasional pada hari kerja dibandingkan hari libur/akhir pekan sepanjang periode 2011–2012?

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
day_types = ['Working Day', 'Weekend / Holiday']
titles = [
    'Hari Kerja (Working Day)\nPola Komuter Bimodal (Puncak Pukul 08:00 & 17:00)',
    'Akhir Pekan & Libur (Weekend / Holiday)\nPola Rekreasi Unimodal (Puncak Pukul 12:00 - 16:00)'
]

for ax, dtype, title in zip(axes, day_types, titles):
    sub = hourly_agg[hourly_agg['day_type'] == dtype]
    ax.plot(sub['hr'], sub['registered'], marker='o', linewidth=2.5, markersize=5, color='#1E88E5', label='Registered User')
    ax.plot(sub['hr'], sub['casual'], marker='s', linewidth=2.5, markersize=5, color='#FB8C00', label='Casual User')
    ax.fill_between(sub['hr'], sub['registered'], color='#1E88E5', alpha=0.08)
    ax.fill_between(sub['hr'], sub['casual'], color='#FB8C00', alpha=0.10)
    
    ax.set_title(title, fontsize=11.5, fontweight='bold', pad=12, color='#212121')
    ax.set_xlabel('Jam Operasional (00:00 - 23:00)', fontsize=10, color='#424242')
    ax.set_xticks(range(0, 24, 2))
    ax.set_ylim(0, 540)
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='#BDBDBD')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].annotate('Puncak Pagi (08:00)\n454,7 registered', xy=(8, 454.74), xytext=(3.0, 470),
                 arrowprops=dict(arrowstyle='->', color='#1E88E5', lw=1.2), fontsize=9, fontweight='bold', color='#1565C0')
axes[0].annotate('Puncak Sore (17:00)\n468,4 registered', xy=(17, 468.39), xytext=(11.2, 495),
                 arrowprops=dict(arrowstyle='->', color='#1E88E5', lw=1.2), fontsize=9, fontweight='bold', color='#1565C0')
axes[1].annotate('Puncak Siang Casual (14:00)\n139,6 casual', xy=(14, 139.59), xytext=(15.2, 55),
                 arrowprops=dict(arrowstyle='->', color='#FB8C00', lw=1.2), fontsize=9, fontweight='bold', color='#EF6C00')

axes[0].set_ylabel('Rata-Rata Penyewaan per Jam (Unit Sepeda)', fontsize=10, color='#424242')
axes[1].legend(frameon=False, fontsize=10, loc='upper right')
fig.suptitle('Perbandingan Pola Penyewaan Sepeda per Jam: Registered vs Casual (2011-2012)', fontsize=13.5, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### Pertanyaan 2: Seberapa besar pengaruh perubahan kondisi cuaca dan musim terhadap rata-rata jumlah penyewaan sepeda harian pada tahun 2011 dan 2012?

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

weather_order = ['Clear / Partly Cloudy', 'Mist / Cloudy', 'Light Snow / Light Rain']
w_means = day_clean.groupby('weather_label')['cnt'].mean().reindex(weather_order)
w_colors = ['#1E88E5', '#B0BEC5', '#E53935']

bars1 = axes[0].bar(w_means.index, w_means.values, color=w_colors, width=0.55)
axes[0].set_title('Rata-Rata Penyewaan Harian Berdasarkan Cuaca\n(Turun 63,02% Saat Hujan Ringan / Salju)', fontsize=11.5, fontweight='bold', pad=12)
axes[0].set_ylabel('Rata-Rata Penyewaan Harian (Unit Sepeda)', fontsize=10)
axes[0].set_ylim(0, 6200)
axes[0].grid(axis='y', linestyle='--', alpha=0.4, color='#BDBDBD')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['Clear / Partly Cloudy', 'Mist / Cloudy', 'Light Snow / Rain'], rotation=0, ha='center', fontsize=9.5)

for bar in bars1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 130, f'{h:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#212121')

s_means = day_clean.groupby('season_label')['cnt'].mean().reindex(season_order)
s_colors = ['#E53935', '#B0BEC5', '#1E88E5', '#B0BEC5']

bars2 = axes[1].bar(s_means.index, s_means.values, color=s_colors, width=0.55)
axes[1].set_title('Rata-Rata Penyewaan Harian Berdasarkan Musim\n(Fall Tertinggi 5.644/hari, Spring Terendah 2.604/hari)', fontsize=11.5, fontweight='bold', pad=12)
axes[1].set_ylabel('Rata-Rata Penyewaan Harian (Unit Sepeda)', fontsize=10)
axes[1].set_ylim(0, 6500)
axes[1].grid(axis='y', linestyle='--', alpha=0.4, color='#BDBDBD')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].tick_params(axis='x', labelsize=10)

for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 130, f'{h:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#212121')

fig.suptitle('Dampak Kondisi Cuaca dan Musim Terhadap Permintaan Penyewaan Sepeda (2011-2012)', fontsize=13.5, fontweight='bold', y=1.03)
plt.setp(axes[0].get_xticklabels(), rotation=0, ha='center')
plt.tight_layout()
plt.show()

**Insight:**
- **Visualisasi 1** memperlihatkan dua pola permintaan yang bertolak belakang: pengguna *registered* menggerakkan lonjakan komuter pada pukul **08:00** dan **17:00–18:00** di hari kerja, sedangkan pengguna *casual* mendongkrak permintaan rekreasi pada rentang **10:00–17:00** di akhir pekan.
- **Visualisasi 2** menegaskan bahwa cuaca buruk (*Light Snow / Light Rain*) memangkas permintaan hingga tersisa **36,98%** dari kondisi cuaca cerah (tidak ada satu pun hari penuh dengan kategori *Heavy Rain / Snow* di `day.csv`, hanya tercatat 3 jam ekstrem di `hour.csv` dengan rata-rata 74,33 sewa/jam). Secara musiman, transisi dari **Spring** ke **Fall** meningkatkan rata-rata permintaan harian lebih dari dua kali lipat (**+116,74%**).

## Analisis Lanjutan (Clustering: Manual Grouping & Binning)

**Tujuan Analisis:**
Untuk memperdalam segmentasi operasional tanpa menggunakan algoritma *machine learning*, diterapkan dua metode pengelompokan berbasis aturan bisnis (*rule-based clustering*):
1. **Manual Grouping (Segmentasi Periode Waktu Operasional):** Mengelompokkan 24 jam operasional pada `hour_clean` menjadi 5 zona waktu aktivitas (*Night*, *Morning Commute*, *Midday*, *Evening Commute*, *Late Evening*) untuk mengukur beban operasional dan komposisi segmen pelanggan di tiap blok waktu.
2. **Binning (Klasifikasi Tingkat Permintaan Harian):** Menggunakan teknik `pd.cut()` untuk membagi volume penyewaan harian (`cnt`) ke dalam 3 tingkatan kapasitas (*Low Demand ≤ 3.000*, *Medium Demand 3.001–6.000*, dan *High Demand > 6.000*) guna memetakan distribusi beban puncak di setiap musim.

In [12]:
def group_time_period(hr):
    if 0 <= hr <= 5:
        return '1. Night (00-05)'
    elif 6 <= hr <= 9:
        return '2. Morning Commute (06-09)'
    elif 10 <= hr <= 15:
        return '3. Midday (10-15)'
    elif 16 <= hr <= 19:
        return '4. Evening Commute (16-19)'
    else:
        return '5. Late Evening (20-23)'

hour_clean['time_period'] = hour_clean['hr'].apply(group_time_period)

period_pivot = hour_clean.groupby(['time_period', 'day_type'])['cnt'].mean().unstack().round(2)
print('Rata-Rata Penyewaan per Jam Berdasarkan Blok Waktu Operasional:')
print(period_pivot)

day_clean['demand_tier'] = pd.cut(
    day_clean['cnt'],
    bins=[0, 3000, 6000, 10000],
    labels=['Low Demand (<=3.000)', 'Medium Demand (3.001-6.000)', 'High Demand (>6.000)']
)

tier_season_pct = (pd.crosstab(day_clean['season_label'], day_clean['demand_tier'], normalize='index').reindex(season_order) * 100).round(2)
print('\nDistribusi Tingkat Permintaan Harian per Musim (%):')
print(tier_season_pct)

Rata-Rata Penyewaan per Jam Berdasarkan Blok Waktu Operasional:
day_type                    Weekend / Holiday  Working Day
time_period                                               
1. Night (00-05)                        43.00        16.44
2. Morning Commute (06-09)              85.00       277.91
3. Midday (10-15)                      338.95       179.66
4. Evening Commute (16-19)             297.34       414.75
5. Late Evening (20-23)                129.70       165.78

Distribusi Tingkat Permintaan Harian per Musim (%):
demand_tier   Low Demand (<=3.000)  Medium Demand (3.001-6.000)  High Demand (>6.000)
season_label                                                                         
Spring                       64.64                        32.60                  2.76
Summer                       13.59                        52.72                 33.70
Fall                          2.13                        56.38                 41.49
Winter                       14.61      

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

x = np.arange(len(period_pivot.index))
width = 0.36
axes[0].bar(x - width/2, period_pivot['Working Day'], width, label='Working Day', color='#1E88E5')
axes[0].bar(x + width/2, period_pivot['Weekend / Holiday'], width, label='Weekend / Holiday', color='#FB8C00')
axes[0].set_title('Rata-Rata Penyewaan per Jam Menurut Zona Waktu', fontsize=11.5, fontweight='bold', pad=12)
axes[0].set_xticks(x)
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_xticklabels(['Night (00-05)', 'Morning (06-09)', 'Midday (10-15)', 'Evening (16-19)', 'Late Eve (20-23)'], rotation=0, ha='center', fontsize=8.8)
axes[0].set_ylabel('Rata-Rata Penyewaan per Jam', fontsize=10)
axes[0].set_ylim(0, 480)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].legend(frameon=False)

tier_colors = ['#E53935', '#90A4AE', '#1E88E5']
tier_season_pct.plot(kind='bar', stacked=True, ax=axes[1], color=tier_colors, width=0.55)
axes[1].set_title('Proporsi Kategori Permintaan Harian per Musim (%)', fontsize=11.5, fontweight='bold', pad=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('Persentase Hari (%)', fontsize=10)
axes[1].set_xticklabels(season_order, rotation=0, fontsize=10)
axes[1].set_ylim(0, 118)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].legend(frameon=False, loc='upper right', fontsize=8.8)

plt.setp(axes[0].get_xticklabels(), rotation=0, ha='center')
plt.tight_layout()
plt.show()

**Insight:**
- **Zona Waktu Paling Padat:** Pada hari kerja, blok **Evening Commute (16:00–19:59)** mencatat beban tertinggi (**414,75 sewa/jam**), disusul **Morning Commute (06:00–09:59)** sebesar **277,91 sewa/jam**. Pada akhir pekan, pusat kepadatan bergeser ke blok **Midday (10:00–15:59)** dengan rata-rata **338,95 sewa/jam**.
- **Distribusi Kapasitas Musiman:** Sebanyak **41,49%** hari pada musim **Fall** dan **33,70%** pada musim **Summer** masuk kategori *High Demand (> 6.000 sewa/hari)*, sedangkan pada musim **Spring**, **64,64%** harinya berada pada kategori *Low Demand (≤ 3.000 sewa/hari)*.

## Conclusion & Recommendation

- **Conclusion pertanyaan 1:** Pengguna *registered* merupakan tulang punggung bisnis dengan kontribusi **81,17%** (2.672.662 transaksi) dan menunjukkan pola komuter dua puncak (*bimodal*) pada hari kerja, yaitu pukul **08:00** (477,01 sewa/jam) dan **17:00–18:00** (525,29 dan 492,23 sewa/jam). Sebaliknya, pengguna *casual* berkontribusi **18,83%** (620.017 transaksi) tetapi meningkat **+126,05%** pada akhir pekan/hari libur (dari 606,57 menjadi 1.371,13 sewa/hari) dengan puncak tunggal (*unimodal*) pada siang hingga sore hari (**12:00–16:00**, tertinggi pukul **13:00** sebesar 372,73 sewa/jam).
- **Conclusion pertanyaan 2:** Kondisi cuaca dan pergantian musim berpengaruh signifikan terhadap volume penyewaan. Dibandingkan hari cerah (*Clear / Partly Cloudy*, rata-rata **4.876,79 sewa/hari**), permintaan turun **17,24%** saat cuaca berkabut/berawan (**4.035,86 sewa/hari**) dan merosot **63,02%** saat hujan ringan/salju (**1.803,29 sewa/hari**). Secara musiman, **Fall** merupakan musim tersibuk (**5.644,30 sewa/hari** dengan 41,49% hari berkategori *High Demand*), sementara **Spring** memiliki permintaan terendah (**2.604,13 sewa/hari**, turun 53,86% dibanding Fall).

**Rekomendasi Action Item:**
1. **Dynamic Fleet Rebalancing Berbasis Zona Waktu & Tipe Hari:** Pada hari kerja, prioritaskan distribusi ulang (*rebalancing*) stok sepeda ke stasiun dekat kawasan perkantoran, kampus, dan transit transportasi sebelum pukul **07:00** dan sebelum pukul **16:00**. Pada akhir pekan, geser alokasi armada menuju taman kota, kawasan wisata, dan jalur rekreasi sebelum pukul **10:00**.
2. **Kampanye Konversi *Casual* ke *Registered* di Akhir Pekan:** Tawarkan promo paket *Weekend-to-Weekday Pass* atau diskon registrasi keanggotaan melalui aplikasi/kios pada pukul **11:00–16:00** di akhir pekan, tepat saat volume pengguna *casual* berada di titik tertinggi.
3. **Penjadwalan *Preventive Maintenance* & Efisiensi Operasional:** Alokasikan jadwal perawatan besar armada sepeda dan stasiun docking pada musim **Spring** (di mana 64,64% hari berada pada level *Low Demand*) serta pada zona waktu **Night (00:00–05:59)** atau saat prakiraan cuaca menunjukkan hujan ringan/salju, sehingga seluruh unit siap beroperasi 100% saat memasuki musim puncak **Summer** dan **Fall**.